# Tarea 1: Análisis Exploratorio y Preparación de Datos

## 1. Carga de datos con Pandas

In [1]:
import pandas as pd

# Carga de los tres subconjuntos del dataset MLSUM en español
df_train = pd.read_csv('../MLSUM_dataset/es_train.csv')
df_val = pd.read_csv('../MLSUM_dataset/es_validation.csv')
df_test = pd.read_csv('../MLSUM_dataset/es_test.csv')

In [2]:
# Inspección rápida de las primeras filas y columnas (text, summary, topic)
print(f"Entrenamiento: {df_train.shape[0]} ejemplos")
print(df_train.head())

Entrenamiento: 266367 ejemplos
                                                text  \
0  De momento, no podemos responder a la pregunta...   
1  Los vuelos han venido registrando este viernes...   
2  El Gobierno turco ha anunciado que emprenderá ...   
3  La policía de Finlandia ha informado este vier...   
4  "Hemos descubierto un agujero vertical en la L...   

                                             summary  \
0  Sofres no ofrece datos por ser festivo.- Telec...   
1  El aeropuerto ha estado hasta las 15.00 con só...   
2  El origen de la leyenda, el san Nicolás histór...   
3  El hombre mató a su ex pareja, a cuatro person...   
4  El hueco en el subsuelo podría ofrecer refugio...   

                      topic  \
0         elpais actualidad   
1         elpais actualidad   
2         elpais actualidad   
3  internacional actualidad   
4       sociedad actualidad   

                                                 url  \
0  http://elpais.com/elpais/2010/01/01/actualidad...

## 2. Validación de la estructura

In [3]:
# Verificación de nulos: Asegúrate de que no haya noticias (text) o resúmenes (summary) vacíos
print("Nulos en el conjunto de entrenamiento:")
print(df_train[['text', 'summary']].isnull().sum())
print("\nNulos en el conjunto de validación:")
print(df_val[['text', 'summary']].isnull().sum())
print("\nNulos en el conjunto de prueba:")
print(df_test[['text', 'summary']].isnull().sum())

Nulos en el conjunto de entrenamiento:
text       0
summary    0
dtype: int64

Nulos en el conjunto de validación:
text       0
summary    0
dtype: int64

Nulos en el conjunto de prueba:
text       0
summary    0
dtype: int64


In [4]:
# Exploración de tópicos: Revisa la columna topic para ver si el dataset está balanceado
print("Distribución de tópicos en el conjunto de entrenamiento:")
print(df_train['topic'].value_counts())
print("\nDistribución de tópicos en el conjunto de validación:")
print(df_val['topic'].value_counts())
print("\nDistribución de tópicos en el conjunto de prueba:")
print(df_test['topic'].value_counts())

Distribución de tópicos en el conjunto de entrenamiento:
topic
elpais opinion                                  22774
cultura actualidad                              21361
ccaa catalunya                                  20097
politica actualidad                             19318
economia actualidad                             18646
                                                ...  
elpais mama_ya_lo_sabe_y_la_abuela_lo_intuia        1
brasil opinion                                      1
cat ciencia                                         1
cat economia                                        1
cat catalunya                                       1
Name: count, Length: 201, dtype: int64

Distribución de tópicos en el conjunto de validación:
topic
elpais opinion                     1573
politica actualidad                1031
economia actualidad                 854
ccaa madrid                         780
ccaa catalunya                      727
                                   ... 
soc

## 3. Preparación para Transformers

In [5]:
%pip install datasets

from datasets import Dataset, DatasetDict

# Conversión de Pandas a Dataset de Hugging Face
train_ds = Dataset.from_pandas(df_train)
val_ds = Dataset.from_pandas(df_val)
test_ds = Dataset.from_pandas(df_test)

# Organización en un único objeto para facilitar el entrenamiento
raw_datasets = DatasetDict({
    'train': train_ds,
    'validation': val_ds,
    'test': test_ds
})

print(raw_datasets)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'topic', 'url', 'title', 'date'],
        num_rows: 266367
    })
    validation: Dataset({
        features: ['text', 'summary', 'topic', 'url', 'title', 'date'],
        num_rows: 10358
    })
    test: Dataset({
        features: ['text', 'summary', 'topic', 'url', 'title', 'date'],
        num_rows: 13920
    })
})


## 4. Modelos generativos


En esta fase evaluaremos la capacidad de los modelos basados en la arquitectura **Transformer (Encoder-Decoder)** para generar resúmenes redactados con sus propias palabras. 

Abordaremos el problema desde dos perspectivas:
1. **Inferencia Directa (Off-the-shelf / Zero-Shot):** Uso de un modelo ya entrenado por la comunidad para resumir en español.
2. **Fine-Tuning (Ajuste Fino):** Entrenamiento de un modelo base multilingüe (`mT5-small`) utilizando nuestro propio conjunto de entrenamiento (`df_train`) para adaptarlo al estilo de "El País".

In [6]:
# Asegúrate de ejecutar esto si estás en un entorno nuevo o en Google Colab
!pip install transformers datasets evaluate rouge_score accelerate
%pip install transformers
%pip install transformers torch sentencepiece

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import torch
import transformers
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

PyTorch version: 2.11.0+cpu
Transformers version: 5.6.2


## Opción 1: Inferencia con un Modelo Pre-entrenado (Zero-Shot)

Para nuestro enfoque generativo utilizaremos un modelo robusto y sin fallos de pesos: **`mT5_multilingual_XLSum`**. Este modelo ha sido afinado específicamente para generar resúmenes periodísticos en decenas de idiomas, incluido el español.

Al cargar explícitamente el `Tokenizador` y el `Modelo` evitamos comportamientos de "caja negra" y tenemos control total sobre hiperparámetros de generación como el *Beam Search*.

In [8]:
# PASO 1: Instalar todas las dependencias necesarias
# Forzamos la reinstalación para asegurar que el entorno esté limpio.
# - `accelerate` optimiza el uso de CPU/GPU.
# - `protobuf` y `sentencepiece` son vitales para el tokenizador de mT5.
%pip install --force-reinstall transformers datasets evaluate rouge_score accelerate protobuf sentencepiece

  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/5d/95/0b0218149b0d6f14df35f5b8f676fa83df4f19ed253c3cc447107ef86eca/transformers-5.6.2-py3-none-any.whl.metadata
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Obtaining dependency information for datasets from https://files.pythonhosted.org/packages/b0/e5/247d094108e42ac26363ab8dc57f168840cf7c05774b40ffeb0d78868fcc/datasets-4.8.4-py3-none-any.whl.metadata
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Obtaining dependency information for evaluate from https://files.pythonhosted.org/packages/3e/af/3e990d8d4002bbc9342adb4facd59506e653da93b2417de0fa6027cb86b1/evaluate-0.4.6-py3-none-any.whl.metadata
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Obtaining dependency information for accelerate from https://files.pythonhosted.org/packages/7e/46/02ac5e262d4af18054b3e922b2baedbb2a03289ee

ERROR: Could not install packages due to an OSError: [WinError 5] Acceso denegado: 'C:\\Users\\jcaba\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\~.hash\\_xxhash.cp312-win_amd64.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# PASO 2: Cargar el modelo y tokenizador
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Forzamos el uso del tokenizador "lento" (use_fast=False) que depende 
# de SentencePiece y Protobuf. Esto es más estable si hay conflictos 
# con los tokenizadores rápidos (basados en Rust).
nombre_modelo_hf = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo_hf, use_fast=False)

# Cargamos el modelo
model = AutoModelForSeq2SeqLM.from_pretrained(nombre_modelo_hf)

print("¡Modelo y tokenizador cargados con éxito!")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


¡Modelo y tokenizador cargados con éxito!


In [10]:
# PASO 3: Generar el resumen
import time

# Configurar dispositivo y mover el modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Usando dispositivo: {device}")

# Preparar datos de ejemplo
texto_ejemplo = df_test['text'].iloc[0]
resumen_real = df_test['summary'].iloc[0]
texto_recortado = texto_ejemplo[:2000] 

# Generación
print("\nGenerando resumen...")
inicio = time.time()

inputs = tokenizer(
    texto_recortado, 
    return_tensors="pt", 
    max_length=512, 
    truncation=True
).to(device)

outputs = model.generate(
    inputs["input_ids"], 
    max_length=150, 
    num_beams=4, 
    early_stopping=True
)

resumen_generado_hf = tokenizer.decode(outputs[0], skip_special_tokens=True)
fin = time.time()

# --- Mostrar Resultados ---
print("="*60)
print(f"RESUMEN GENERADO con mT5-XLSum (Tardó {fin-inicio:.2f} s):")
print("="*60)
print(resumen_generado_hf + "\n")

print("="*60)
print("RESUMEN DE REFERENCIA (Ground Truth):")
print("="*60)
print(resumen_real)

Usando dispositivo: cpu

Generando resumen...
RESUMEN GENERADO con mT5-XLSum (Tardó 5.82 s):
Hace un año, Mariano Rajoy anunció su salida de la presidencia de España.

RESUMEN DE REFERENCIA (Ground Truth):
Doce meses después de la moción de censura, sus protagonistas, que entonces dudaron, ven claro que fue un acierto


## Evaluación Cuantitativa del Modelo Generativo (Zero-Shot)
Vamos a calcular las métricas ROUGE para el modelo pre-entrenado de Hugging Face y compararlas con el Baseline Extractivo (TF-IDF).

In [11]:
import evaluate
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Configurar dispositivo (Detectará automáticamente tu CPU)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {dispositivo}")

# 2. Cargar modelo y tokenizador manualmente
nombre_modelo_hf = "csebuetnlp/mT5_multilingual_XLSum"
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo_hf)
model = AutoModelForSeq2SeqLM.from_pretrained(nombre_modelo_hf).to(dispositivo)

rouge = evaluate.load('rouge')

# Seleccionamos muestra de 100 noticias
num_ejemplos_gen = 100 
df_muestra_gen = df_test.head(num_ejemplos_gen)

predicciones_gen = []
referencias_gen = df_muestra_gen['summary'].tolist()

print(f"\nGenerando resúmenes abstractivos para {num_ejemplos_gen} noticias...")

for texto in tqdm(df_muestra_gen['text']):
    texto_recortado = texto[:2000] 
    
    try:
        # Generación manual
        inputs = tokenizer(texto_recortado, return_tensors="pt", max_length=512, truncation=True).to(dispositivo)
        
        outputs = model.generate(
            inputs["input_ids"], 
            max_length=100, 
            min_length=20, 
            num_beams=4, 
            early_stopping=True
        )
        resumen = tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"Error procesando un texto: {e}")
        resumen = ""
        
    predicciones_gen.append(resumen)

print("\nCalculando métricas ROUGE...")
resultados_rouge_gen = rouge.compute(predictions=predicciones_gen, references=referencias_gen)

print("="*60)
print(" RESULTADOS ROUGE - TRANSFORMER GENERATIVO (Opción A)")
print("="*60)
print(f"ROUGE-1 (Unigramas / Conceptos): {resultados_rouge_gen['rouge1']*100:.2f} %")
print(f"ROUGE-2 (Bigramas / Frases):    {resultados_rouge_gen['rouge2']*100:.2f} %")
print(f"ROUGE-L (Estructura general):   {resultados_rouge_gen['rougeL']*100:.2f} %")
print("="*60)

Usando dispositivo: cpu


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Generando resúmenes abstractivos para 100 noticias...


100%|██████████| 100/100 [06:32<00:00,  3.93s/it]



Calculando métricas ROUGE...
 RESULTADOS ROUGE - TRANSFORMER GENERATIVO (Opción A)
ROUGE-1 (Unigramas / Conceptos): 22.16 %
ROUGE-2 (Bigramas / Frases):    5.43 %
ROUGE-L (Estructura general):   17.41 %


## Opción B: Fine-Tuning de nuestro propio Modelo (mT5)

Ahora vamos a aplicar el **Aprendizaje Supervisado**. Tomaremos un modelo multilingüe base (`google/mt5-small`) y le enseñaremos a resumir usando nuestros pares de `text` y `summary` de la variable `raw_datasets` que creamos en la Fase 1.

**⚠️ ADVERTENCIA:** El entrenamiento de Transformers requiere mucha computación.

In [12]:
# Tokenización para Seq2Seq
from transformers import AutoTokenizer

# Usamos mT5-small porque es ligero e ideal para prácticas académicas
model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Dimensiones máximas basadas en nuestro análisis exploratorio (EDA)
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    # 1. Tokenizamos las noticias (inputs)
    inputs = [doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # 2. Tokenizamos los resúmenes (labels/objetivo)
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizando el dataset (Esto puede tardar unos minutos)...")
# Usamos el DatasetDict 'raw_datasets' que definiste en tu notebook anterior
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Tokenizando el dataset (Esto puede tardar unos minutos)...


Map:   0%|          | 0/266367 [00:00<?, ? examples/s]

Map:   0%|          | 0/10358 [00:00<?, ? examples/s]

Map:   0%|          | 0/13920 [00:00<?, ? examples/s]

In [13]:
%pip install "accelerate>=1.1.0" "transformers[torch]" -U

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# Configuración del Entrenamiento - Trainer
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# 1. Cargamos la arquitectura Encoder-Decoder del modelo
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 2. Data Collator: Se encarga de hacer el padding dinámico de los tensores
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 3. Argumentos de entrenamiento (Ajuste de hiperparámetros)
batch_size = 4

args = Seq2SeqTrainingArguments(
    output_dir="./resultados_mt5_mlsum",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3, # Con 3 épocas suele ser suficiente para notar aprendizaje
    predict_with_generate=True,
    fp16=False, # Ponlo a True solo si estás usando una GPU moderna
)

# 4. Inicializamos el Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"].select(range(5000)), # Usamos solo 5000 ejemplos para no tardar días
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    data_collator=data_collator,
)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [16]:
# Ejecutar el Fine-Tuning

print("Iniciando el ajuste fino (Fine-Tuning)...")
trainer.train()

# Guardar el modelo entrenado
trainer.save_model("mi_modelo_resumen_mt5")

Iniciando el ajuste fino (Fine-Tuning)...


c:\Users\jcaba\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,4.878739,2.944453
2,4.093690,2.853430
3,3.975360,2.843020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\jcaba\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Evaluación del Modelo Propio (Fine-Tuned)
Una vez finalizado el entrenamiento, cargamos nuestro modelo personalizado desde el disco local (`mi_modelo_resumen_mt5`) para evaluar su rendimiento. Este modelo partió de `mT5-small` y ha aprendido el estilo específico de los resúmenes de "El País".

In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import time
import os

# Ruta donde guardamos el modelo en el paso anterior
ruta_modelo_propio = "mi_modelo_resumen_mt5"

if not os.path.exists(ruta_modelo_propio):
    print(f"⚠️ ERROR: No se encuentra la carpeta '{ruta_modelo_propio}'. Asegúrate de que el entrenamiento ha terminado.")
else:
    print(f"Cargando nuestro modelo afinado desde: {ruta_modelo_propio}...")
    
    # Cargamos el modelo y el tokenizador que hemos entrenado nosotros mismos
    tokenizer_ft = AutoTokenizer.from_pretrained(ruta_modelo_propio, use_fast=False)
    model_ft = AutoModelForSeq2SeqLM.from_pretrained(ruta_modelo_propio)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_ft.to(device)
    
    # Prueba con la primera noticia de test
    texto_ejemplo = df_test['text'].iloc[0]
    resumen_real = df_test['summary'].iloc[0]
    texto_recortado = texto_ejemplo[:2000]
    
    print("\nGenerando resumen con nuestro modelo propio...")
    inicio = time.time()
    
    inputs_ft = tokenizer_ft(texto_recortado, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    outputs_ft = model_ft.generate(
        inputs_ft["input_ids"], 
        max_length=150, 
        num_beams=4, 
        early_stopping=True
    )
    
    resumen_generado_ft = tokenizer_ft.decode(outputs_ft[0], skip_special_tokens=True)
    fin = time.time()
    
    print("="*60)
    print(f"RESUMEN GENERADO (Modelo Fine-Tuned) (Tardó {fin-inicio:.2f} s):")
    print("="*60)
    print(resumen_generado_ft + "\n")
    
    print("="*60)
    print("RESUMEN DE REFERENCIA (Ground Truth):")
    print("="*60)
    print(resumen_real)

Cargando nuestro modelo afinado desde: mi_modelo_resumen_mt5...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Generando resumen con nuestro modelo propio...
RESUMEN GENERADO (Modelo Fine-Tuned) (Tardó 1.95 s):
La moción de censura de Pedro Sánchez y Albert Rivera

RESUMEN DE REFERENCIA (Ground Truth):
Doce meses después de la moción de censura, sus protagonistas, que entonces dudaron, ven claro que fue un acierto


### Evaluación Cuantitativa de la Opción B (Métrica ROUGE)
Calculamos las métricas ROUGE para nuestro modelo sobre la misma muestra de 100 noticias. El objetivo es comparar si el *Fine-Tuning* supera al modelo pre-entrenado generalista (Opción A) y al Baseline extractivo (TF-IDF).

In [18]:
import evaluate
from tqdm import tqdm

if os.path.exists(ruta_modelo_propio):
    rouge = evaluate.load('rouge')
    
    num_ejemplos_gen = 100 
    df_muestra_gen = df_test.head(num_ejemplos_gen)
    
    predicciones_ft = []
    referencias_ft = df_muestra_gen['summary'].tolist()
    
    print(f"\nGenerando resúmenes con el modelo FINE-TUNED para {num_ejemplos_gen} noticias...")
    
    for texto in tqdm(df_muestra_gen['text']):
        texto_recortado = texto[:2000] 
        
        try:
            inputs_ft = tokenizer_ft(texto_recortado, return_tensors="pt", max_length=512, truncation=True).to(device)
            
            outputs_ft = model_ft.generate(
                inputs_ft["input_ids"], 
                max_length=100, 
                min_length=20, 
                num_beams=4, 
                early_stopping=True
            )
            resumen = tokenizer_ft.decode(outputs_ft[0], skip_special_tokens=True)
        except Exception as e:
            print(f"Error procesando un texto: {e}")
            resumen = ""
            
        predicciones_ft.append(resumen)
    
    print("\nCalculando métricas ROUGE...")
    resultados_rouge_ft = rouge.compute(predictions=predicciones_ft, references=referencias_ft)
    
    print("="*60)
    print(" RESULTADOS ROUGE - TRANSFORMER PROPIO (Opción B)")
    print("="*60)
    print(f"ROUGE-1 (Unigramas / Conceptos): {resultados_rouge_ft['rouge1']*100:.2f} %")
    print(f"ROUGE-2 (Bigramas / Frases):    {resultados_rouge_ft['rouge2']*100:.2f} %")
    print(f"ROUGE-L (Estructura general):   {resultados_rouge_ft['rougeL']*100:.2f} %")
    print("="*60)
else:
    print("Ejecuta esta celda cuando el modelo haya terminado de entrenar.")


Generando resúmenes con el modelo FINE-TUNED para 100 noticias...


100%|██████████| 100/100 [02:47<00:00,  1.67s/it]



Calculando métricas ROUGE...
 RESULTADOS ROUGE - TRANSFORMER PROPIO (Opción B)
ROUGE-1 (Unigramas / Conceptos): 21.77 %
ROUGE-2 (Bigramas / Frases):    5.79 %
ROUGE-L (Estructura general):   17.34 %
